# Maia Lite 355M v3 — Clean Pretraining

Clean, reproducible pretraining pipeline for the Maia Lite base model.

**Fixed architecture**
- GPT-2 Medium–matched decoder-only Transformer
- 354,823,168 parameters
- 24 layers, 16 heads, 1,024 embedding dimension
- 50,257-token vocabulary
- 1,024-token context

**Training budget**
- Chinchilla reference: ~20 tokens/parameter
- Exact reference: 7,096,463,360 tokens
- Batch-aligned target: 7,096,467,456 tokens
- Effective batch: 32 sequences = 32,768 tokens/optimizer step
- Target: 216,567 optimizer steps

This notebook intentionally does **not** reuse the old Maia tokenizer, corpus, model weights, samples, or materialization state. FineMath has been replaced by OpenWebMath.


In [ ]:
# 0. INSTALL / MOUNT
!pip -q install -U datasets transformers tokenizers accelerate

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. GLOBAL SPECIFICATION

import os
import gc
import json
import math
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3")
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
TOKENIZER_SAMPLE_DIR = PROJECT_DIR / "tokenizer_sample"
CORPUS_DIR = PROJECT_DIR / "corpus"
TRAIN_DIR = CORPUS_DIR / "train"
STATE_DIR = CORPUS_DIR / "state"
MANIFEST_DIR = CORPUS_DIR / "manifests"
VALIDATION_DIR = PROJECT_DIR / "validation"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for p in [
    PROJECT_DIR, TOKENIZER_DIR, TOKENIZER_SAMPLE_DIR,
    CORPUS_DIR, TRAIN_DIR, STATE_DIR, MANIFEST_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 50_257
SEQUENCE_LENGTH = 1_024
N_PARAMS_REFERENCE = 354_823_168
CHINCHILLA_TOKENS = 20 * N_PARAMS_REFERENCE

MICRO_BATCH = 16
GRAD_ACCUM = 2
EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM
TOKENS_PER_STEP = EFFECTIVE_BATCH * SEQUENCE_LENGTH

TARGET_STEPS = math.ceil(CHINCHILLA_TOKENS / TOKENS_PER_STEP)
TARGET_TOTAL_TOKENS = TARGET_STEPS * TOKENS_PER_STEP

# 8 Mi tokens = 16 MiB in uint16, divisible by 1024.
SHARD_TOKENS = 8_388_608
DTYPE = np.uint16

print(f"Reference parameters:     {N_PARAMS_REFERENCE:,}")
print(f"Chinchilla target:        {CHINCHILLA_TOKENS:,} tokens")
print(f"Tokens/optimizer step:    {TOKENS_PER_STEP:,}")
print(f"Target optimizer steps:   {TARGET_STEPS:,}")
print(f"Batch-aligned target:     {TARGET_TOTAL_TOKENS:,} tokens")
print(f"Difference from 20N:      {TARGET_TOTAL_TOKENS - CHINCHILLA_TOKENS:,} tokens")

Reference parameters:     354,823,168
Chinchilla target:        7,096,463,360 tokens
Tokens/optimizer step:    32,768
Target optimizer steps:   216,567
Batch-aligned target:     7,096,467,456 tokens
Difference from 20N:      4,096 tokens


In [ ]:
# 2. CORPUS SPECIFICATION
# Same high-quality source families validated in the previous experiment,
# but rebuilt from raw documents with the NEW tokenizer.
#
# Weights sum to 1.0:
#   EN 45%, PT 22.5%, ES 22.5%, Code 5%, Math 3%, Science 2%.

SOURCES = [
    {
        "id": "fineweb_edu_en",
        "name": "FineWeb-Edu EN",
        "repo": "HuggingFaceFW/fineweb-edu",
        "config": "sample-10BT",
        "split": "train",
        "weight": 0.45,
        "text_field": "text",
    },
    {
        "id": "fineweb2_hq_pt",
        "name": "FineWeb2-HQ PT",
        "repo": "epfml/FineWeb2-HQ",
        "config": "por_Latn",
        "split": "train",
        "weight": 0.225,
        "text_field": "text",
    },
    {
        "id": "fineweb2_hq_es",
        "name": "FineWeb2-HQ ES",
        "repo": "epfml/FineWeb2-HQ",
        "config": "spa_Latn",
        "split": "train",
        "weight": 0.225,
        "text_field": "text",
    },
    {
        "id": "opencoder",
        "name": "OpenCoder FineWeb Code",
        "repo": "OpenCoder-LLM/opc-fineweb-code-corpus",
        "config": None,
        "split": "train",
        "weight": 0.05,
        "text_field": "text",
    },
    {
        "id": "openwebmath",
        "name": "OpenWebMath",
        "repo": "open-web-math/open-web-math",
        "config": None,
        "split": "train",
        "weight": 0.03,
        "text_field": "text",
    },
    {
        "id": "pes2o",
        "name": "peS2o Scientific",
        "repo": "common-pile/peS2o",
        "config": None,
        "split": "train",
        "weight": 0.02,
        "text_field": "text",
    },
]

assert abs(sum(s["weight"] for s in SOURCES) - 1.0) < 1e-12

# Allocate source targets in whole 1024-token sequences.
remaining_sequences = TARGET_TOTAL_TOKENS // SEQUENCE_LENGTH
for i, source in enumerate(SOURCES):
    if i < len(SOURCES) - 1:
        seqs = int((TARGET_TOTAL_TOKENS // SEQUENCE_LENGTH) * source["weight"])
    else:
        seqs = remaining_sequences
    source["target_sequences"] = seqs
    source["target_tokens"] = seqs * SEQUENCE_LENGTH
    remaining_sequences -= seqs

# Correct any rounding remainder by assigning it to English.
if remaining_sequences:
    SOURCES[0]["target_sequences"] += remaining_sequences
    SOURCES[0]["target_tokens"] += remaining_sequences * SEQUENCE_LENGTH

print(f"{'Source':28s} {'Weight':>8s} {'Tokens':>18s}")
for s in SOURCES:
    print(f"{s['name']:28s} {s['weight']:8.3%} {s['target_tokens']:18,}")
print("-" * 58)
print(f"{'TOTAL':28s} {'100%':>8s} {sum(s['target_tokens'] for s in SOURCES):18,}")

Source                         Weight             Tokens
FineWeb-Edu EN                45.000%      3,193,409,536
FineWeb2-HQ PT                22.500%      1,596,704,768
FineWeb2-HQ ES                22.500%      1,596,704,768
OpenCoder FineWeb Code         5.000%        354,823,168
OpenWebMath                    3.000%        212,893,696
peS2o Scientific               2.000%        141,931,520
----------------------------------------------------------
TOTAL                            100%      7,096,467,456


## Tokenizer

Train a fresh byte-level BPE tokenizer from a **stratified sample of the same source families** used for pretraining.

The special tokens are fixed at IDs 0–4. During corpus materialization, `</s>` is appended **explicitly** after every document. We do not rely on `add_special_tokens=True`.


In [ ]:
# 3. BUILD STRATIFIED TOKENIZER TRAINING SAMPLE
# Run once. The sample is persisted so tokenizer training is reproducible.
#
# 1 GiB of UTF-8 text is enough for a representative 50k BPE vocabulary
# while remaining practical in Colab. Allocation follows corpus weights.

TOKENIZER_SAMPLE_BYTES = 1_073_741_824  # 1 GiB
MIN_DOCUMENT_CHARS = 200

sample_files = []

for source in SOURCES:
    target_bytes = int(TOKENIZER_SAMPLE_BYTES * source["weight"])
    out_path = TOKENIZER_SAMPLE_DIR / f"{source['id']}.txt"
    sample_files.append(str(out_path))

    if out_path.exists() and out_path.stat().st_size >= target_bytes:
        print(f"[SKIP] {source['name']}: existing sample {out_path.stat().st_size:,} bytes")
        continue

    print(f"\nBuilding tokenizer sample: {source['name']}")
    print(f"Target: {target_bytes:,} bytes")

    ds = load_dataset(
        source["repo"],
        source["config"],
        split=source["split"],
        streaming=True,
    )

    written = 0
    documents = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for row in ds:
            text = row.get(source["text_field"])
            if not isinstance(text, str):
                continue
            text = text.strip()
            if len(text) < MIN_DOCUMENT_CHARS:
                continue

            encoded = (text + "\n").encode("utf-8")
            if written + len(encoded) > target_bytes:
                break

            f.write(text)
            f.write("\n")
            written += len(encoded)
            documents += 1

            if documents % 10_000 == 0:
                print(f"  documents={documents:,} bytes={written:,}", end="\r")

    print(f"  DONE documents={documents:,} bytes={out_path.stat().st_size:,}")


Building tokenizer sample: FineWeb-Edu EN
Target: 483,183,820 bytes


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

  DONE documents=100,136 bytes=483,060,078

Building tokenizer sample: FineWeb2-HQ PT
Target: 241,591,910 bytes


README.md:   0%|          | 0.00/9.61k [00:00<?, ?B/s]

...

  DONE documents=4,343 bytes=32,197,786

Building tokenizer sample: peS2o Scientific
Target: 21,474,836 bytes


README.md:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/102 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/102 [00:00<?, ?it/s]

  DONE documents=707 bytes=21,455,210


In [ ]:
# 3B. TOKENIZER SAMPLE PREFLIGHT
print("=" * 80)
print("TOKENIZER SAMPLE PREFLIGHT")
print("=" * 80)
total_bytes = 0
for filename in sample_files:
    path = Path(filename)
    assert path.exists(), f"Missing file: {path}"
    size = path.stat().st_size
    assert size > 1_000_000, f"Suspiciously small sample: {path}"
    total_bytes += size
    with open(path, "r", encoding="utf-8") as f:
        preview = f.read(600)
    print(f"\n{path.name}: {size / 1024**2:.2f} MiB")
    print(repr(preview))
print(f"\nTotal sample: {total_bytes / 1024**2:.2f} MiB")
assert total_bytes > 900 * 1024**2
print("PREFLIGHT PASSED")

TOKENIZER SAMPLE PREFLIGHT

fineweb_edu_en.txt: 460.68 MiB
'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Ell'

fineweb2_hq_pt.txt: 230.40 MiB
'Boehner abre as portas para aumentos tributários nos EUA\nPor David Lawder e Mark Felsenthal\nWASHINGTON, 17 Dez (Reuters) - O primeiro movimento real nas negociações sobre o "abismo fiscal" nos Estados Unidos começou no domingo, quando o presidente republicano da Câmara dos Deputados, John Boehner, s

In [ ]:

# ==============================================================================
# 4. TRAIN NEW TOKENIZER - ROBUST VERSION
# ==============================================================================

import os
import shutil
import numpy as np

from pathlib import Path
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast


# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

SPECIAL_TOKENS = [
    "<s>",
    "<pad>",
    "</s>",
    "<unk>",
    "<mask>",
]

print("=" * 80)
print("TRAINING MAIA LITE V3 TOKENIZER")
print("=" * 80)

print(f"Target vocabulary: {VOCAB_SIZE:,}")
print(f"Training files:    {len(sample_files)}")

for filename in sample_files:
    size = Path(filename).stat().st_size

    print(
        f"  {Path(filename).name:<25} "
        f"{size / 1024**2:8.2f} MiB"
    )


# ------------------------------------------------------------------------------
# Remove old tokenizer artifacts
# ------------------------------------------------------------------------------

TOKENIZER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

old_files = [
    "vocab.json",
    "merges.txt",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
]

for filename in old_files:

    path = TOKENIZER_DIR / filename

    if path.exists():
        print("Removing old:", path)
        path.unlink()


# ------------------------------------------------------------------------------
# Train Byte-Level BPE
# ------------------------------------------------------------------------------

print()
print("Training Byte-Level BPE...")
print()

tokenizer_raw = ByteLevelBPETokenizer()

tokenizer_raw.train(
    files=[str(x) for x in sample_files],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)


# ------------------------------------------------------------------------------
# CRITICAL CHECK: inspect BEFORE saving
# ------------------------------------------------------------------------------

raw_vocab = tokenizer_raw.get_vocab()

print()
print("=" * 80)
print("RAW TOKENIZER RESULT")
print("=" * 80)

print(f"Vocabulary: {len(raw_vocab):,}")

for token in SPECIAL_TOKENS:
    print(
        f"{token:<8} -> "
        f"{tokenizer_raw.token_to_id(token)}"
    )

assert len(raw_vocab) == VOCAB_SIZE, (
    f"ByteLevelBPETokenizer produced "
    f"{len(raw_vocab):,} tokens instead of "
    f"{VOCAB_SIZE:,}."
)

assert tokenizer_raw.token_to_id("<s>") == 0
assert tokenizer_raw.token_to_id("<pad>") == 1
assert tokenizer_raw.token_to_id("</s>") == 2
assert tokenizer_raw.token_to_id("<unk>") == 3
assert tokenizer_raw.token_to_id("<mask>") == 4


# ------------------------------------------------------------------------------
# Save vocab + merges
# ------------------------------------------------------------------------------

print()
print("Saving vocab.json and merges.txt...")

tokenizer_raw.save_model(
    str(TOKENIZER_DIR)
)


# ------------------------------------------------------------------------------
# Save COMPLETE tokenizer.json
# ------------------------------------------------------------------------------

TOKENIZER_JSON = TOKENIZER_DIR / "tokenizer.json"

tokenizer_raw._tokenizer.save(
    str(TOKENIZER_JSON)
)

assert TOKENIZER_JSON.exists()

print("Saved:", TOKENIZER_JSON)


# ------------------------------------------------------------------------------
# Load directly from tokenizer.json
# ------------------------------------------------------------------------------

print()
print("Loading GPT2TokenizerFast from tokenizer.json...")

tokenizer = GPT2TokenizerFast(
    tokenizer_file=str(TOKENIZER_JSON),

    bos_token="<s>",
    pad_token="<pad>",
    eos_token="</s>",
    unk_token="<unk>",
    mask_token="<mask>",

    model_max_length=SEQUENCE_LENGTH,
)


# ------------------------------------------------------------------------------
# Save Hugging Face representation
# ------------------------------------------------------------------------------

tokenizer.save_pretrained(
    str(TOKENIZER_DIR)
)


# ------------------------------------------------------------------------------
# Reload FROM DISK
# ------------------------------------------------------------------------------

del tokenizer

tokenizer = GPT2TokenizerFast.from_pretrained(
    str(TOKENIZER_DIR),
    local_files_only=True,
)


# ------------------------------------------------------------------------------
# Final validation
# ------------------------------------------------------------------------------

print()
print("=" * 80)
print("FINAL TOKENIZER")
print("=" * 80)

print(f"Vocabulary: {len(tokenizer):,}")

print(
    "BOS:",
    tokenizer.bos_token,
    tokenizer.bos_token_id,
)

print(
    "PAD:",
    tokenizer.pad_token,
    tokenizer.pad_token_id,
)

print(
    "EOS:",
    tokenizer.eos_token,
    tokenizer.eos_token_id,
)

print(
    "UNK:",
    tokenizer.unk_token,
    tokenizer.unk_token_id,
)

print(
    "MASK:",
    tokenizer.mask_token,
    tokenizer.mask_token_id,
)


# ------------------------------------------------------------------------------
# Hard assertions
# ------------------------------------------------------------------------------

assert len(tokenizer) == VOCAB_SIZE

assert tokenizer.bos_token_id == 0
assert tokenizer.pad_token_id == 1
assert tokenizer.eos_token_id == 2
assert tokenizer.unk_token_id == 3
assert tokenizer.mask_token_id == 4

assert len(tokenizer) <= np.iinfo(
    np.uint16
).max


# ------------------------------------------------------------------------------
# Round-trip test
# ------------------------------------------------------------------------------

TEST_TEXTS = [
    "The theory of general relativity describes gravity.",
    "A inteligência artificial pode auxiliar a pesquisa científica.",
    "La inteligencia artificial es un campo de la informática.",
    "def square(x): return x * x",
    "E = mc²",
]


print()
print("=" * 80)
print("ROUND-TRIP TEST")
print("=" * 80)

for text in TEST_TEXTS:

    ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    decoded = tokenizer.decode(
        ids,
        skip_special_tokens=False,
    )

    print()
    print("TEXT:   ", repr(text))
    print("TOKENS: ", len(ids))
    print("IDS:    ", ids[:30])
    print("DECODED:", repr(decoded))

    assert decoded == text


# ------------------------------------------------------------------------------
# Explicit EOS test
# ------------------------------------------------------------------------------

print()
print("=" * 80)
print("EXPLICIT EOS TEST")
print("=" * 80)

text = "This is document one."

ids = tokenizer(
    text,
    add_special_tokens=False,
)["input_ids"]

# We DO NOT rely on add_special_tokens=True.
ids.append(tokenizer.eos_token_id)

print("IDs:")
print(ids)

print()
print("Tokens:")
print(
    tokenizer.convert_ids_to_tokens(ids)
)

print()
print("Decoded:")
print(
    repr(
        tokenizer.decode(
            ids,
            skip_special_tokens=False,
        )
    )
)

assert ids[-1] == 2

print()
print("=" * 80)
print("TOKENIZER PASSED ALL TESTS")
print("=" * 80)

TRAINING MAIA LITE V3 TOKENIZER
Target vocabulary: 50,257
Training files:    6
  fineweb_edu_en.txt          460.68 MiB
  fineweb2_hq_pt.txt          230.40 MiB
  fineweb2_hq_es.txt          230.40 MiB
  opencoder.txt                51.20 MiB
  openwebmath.txt              30.71 MiB
  pes2o.txt                    20.46 MiB

Training Byte-Level BPE...


RAW TOKENIZER RESULT
Vocabulary: 50,257
<s>      -> 0
<pad>    -> 1
</s>     -> 2
<unk>    -> 3
<mask>   -> 4

Saving vocab.json and merges.txt...
Saved: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer/tokenizer.json

Loading GPT2TokenizerFast from tokenizer.json...

FINAL TOKENIZER
Vocabulary: 50,257
BOS: <s> 0
PAD: <pad> 1
EOS: </s> 2
UNK: <unk> 3
MASK: <mask> 4

ROUND-TRIP TEST

TEXT:    'The theory of general relativity describes gravity.'
TOKENS:  8
IDS:     [621, 4388, 303, 1432, 42874, 9500, 14767, 18]
DECODED: 'The theory of general relativity describes gravity.'

TEXT:    'A inteligência artificial pode auxiliar a pesqu

In [ ]:
# 5. TOKENIZER AUDIT
# Critical regression test: explicit EOS must exist in the sequence.

TEST_TEXTS = [
    "Water is a chemical compound composed of hydrogen and oxygen.",
    "The theory of general relativity describes gravity as spacetime curvature.",
    "A computer operating system manages hardware and software resources.",
    "O Brasil é o maior país da América do Sul.",
    "La fotosíntesis convierte la energía luminosa en energía química.",
    "def square(x):\n    return x * x",
]

for text in TEST_TEXTS:
    ids = tokenizer.encode(text, add_special_tokens=False)
    ids_with_eos = ids + [tokenizer.eos_token_id]
    decoded = tokenizer.decode(ids, clean_up_tokenization_spaces=False)

    print("\nTEXT:", repr(text))
    print("TOKENS:", len(ids))
    print("ROUND TRIP:", decoded == text)
    print("LAST TOKEN WITH EXPLICIT EOS:", ids_with_eos[-1], tokenizer.convert_ids_to_tokens([ids_with_eos[-1]]))

    assert decoded == text
    assert ids_with_eos[-1] == tokenizer.eos_token_id

print("\nTokenizer audit: PASS")


TEXT: 'Water is a chemical compound composed of hydrogen and oxygen.'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'The theory of general relativity describes gravity as spacetime curvature.'
TOKENS: 12
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'A computer operating system manages hardware and software resources.'
TOKENS: 10
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'O Brasil é o maior país da América do Sul.'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'La fotosíntesis convierte la energía luminosa en energía química.'
TOKENS: 13
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

TEXT: 'def square(x):\n    return x * x'
TOKENS: 11
ROUND TRIP: True
LAST TOKEN WITH EXPLICIT EOS: 2 ['</s>']

Tokenizer audit: PASS


## Corpus materialization

Each source is materialized independently as deterministic `uint16` shards.

Rules:
1. Read complete documents from the source stream.
2. Tokenize without automatic special tokens.
3. Append EOS explicitly to every accepted document.
4. Concatenate document token streams continuously.
5. Write aligned 1024-token sequences.
6. Never pad training data.
7. Persist document counters and SHA-256 manifests for safe resume.

Streaming is used only as an **input transport** while constructing the fixed corpus. The Trainer never trains directly from a streaming dataset.


In [ ]:
# 6. CORPUS MATERIALIZER — CRASH-SAFE CARRY
# Training and validation are separated by a deterministic SHA-256 partition.

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def is_validation_document(text):
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % 1000 < 5  # ~0.5%

def snapshot_path(source):
    return STATE_DIR / f"{source['id']}_snapshot.npz"

def manifest_path(source):
    return MANIFEST_DIR / f"{source['id']}.json"

def default_state():
    return {
        "documents_seen": 0,
        "documents_accepted": 0,
        "validation_documents_skipped": 0,
        "tokens_written": 0,
        "shards_written": 0,
        "records": [],
        "complete": False,
    }

def load_snapshot(source):
    p = snapshot_path(source)
    if not p.exists():
        return default_state(), np.empty(0, dtype=DTYPE)
    with np.load(p, allow_pickle=False) as z:
        state_json = bytes(z["state_json"].tolist()).decode("utf-8")
        state = json.loads(state_json)
        carry = np.asarray(z["carry"], dtype=DTYPE)
    return state, carry

def save_snapshot(source, state, carry):
    # State + carry are committed as ONE atomic file. A crash can therefore
    # never pair an old state with a new carry (or vice versa).
    p = snapshot_path(source)
    tmp = Path(str(p) + ".tmp.npz")
    state_bytes = json.dumps(state, separators=(",", ":")).encode("utf-8")
    np.savez(
        tmp,
        state_json=np.frombuffer(state_bytes, dtype=np.uint8),
        carry=np.asarray(carry, dtype=DTYPE),
    )
    os.replace(tmp, p)

def atomic_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    os.replace(tmp, path)

def materialize_source(source):
    state, carry = load_snapshot(source)
    # Manifest is derived from the authoritative snapshot and can be regenerated.
    atomic_json(manifest_path(source), state.get("records", []))
    if state["complete"]:
        print(f"[SKIP] {source['name']} already complete."); return
    print("\n"+"="*80); print(source["name"]); print("="*80)
    print(f"Target tokens: {source['target_tokens']:,}")
    print(f"Resume documents_seen: {state['documents_seen']:,}")
    print(f"Resume tokens_written: {state['tokens_written']:,}")

    ds=load_dataset(source["repo"],source["config"],split=source["split"],streaming=True)
    iterator=iter(ds)
    for _ in range(state["documents_seen"]):
        try: next(iterator)
        except StopIteration: raise RuntimeError(f"Source exhausted while resuming {source['name']}")

    # Carry came from the same atomic snapshot as the state.
    buffer=[carry] if len(carry) else []; buffer_tokens=len(carry)
    while state["tokens_written"] < source["target_tokens"]:
        remaining=source["target_tokens"]-state["tokens_written"]
        needed=min(SHARD_TOKENS,remaining)
        assert needed % SEQUENCE_LENGTH == 0
        while buffer_tokens < needed:
            try: row=next(iterator)
            except StopIteration: raise RuntimeError(f"Source exhausted before target: {source['name']}")
            state["documents_seen"] += 1
            text=row.get(source["text_field"])
            if not isinstance(text,str): continue
            text=text.strip()
            if not text: continue
            if is_validation_document(text):
                state["validation_documents_skipped"] += 1; continue
            ids=tokenizer.encode(text,add_special_tokens=False)
            if not ids: continue
            ids.append(tokenizer.eos_token_id)
            arr=np.asarray(ids,dtype=DTYPE); buffer.append(arr); buffer_tokens += len(arr)
            state["documents_accepted"] += 1

        joined=np.concatenate(buffer); shard=joined[:needed]; surplus=joined[needed:]
        shard_index=state["shards_written"]
        path=TRAIN_DIR/f"{source['id']}_{shard_index:05d}.bin"; tmp=Path(str(path)+".tmp")
        shard.tofile(tmp); os.replace(tmp,path)
        record={"shard":path.name,"tokens":int(needed),"sha256":sha256_file(path)}
        state["records"] = state.get("records",[]) + [record]
        state["tokens_written"] += int(needed); state["shards_written"] += 1
        # Commit authoritative resume snapshot first. If the process dies before
        # the human-readable manifest update, resume regenerates the manifest.
        save_snapshot(source, state, surplus)
        atomic_json(manifest_path(source), state["records"])
        buffer=[surplus] if len(surplus) else []; buffer_tokens=len(surplus)
        print(f"shard={shard_index:05d} tokens={state['tokens_written']:,}/{source['target_tokens']:,} docs={state['documents_seen']:,}")

    state["complete"] = state["tokens_written"] == source["target_tokens"]
    save_snapshot(source, state, np.asarray(buffer[0], dtype=DTYPE) if buffer else np.empty(0, dtype=DTYPE))
    atomic_json(manifest_path(source), state["records"])
    assert state["complete"]
    print(f"DONE: {source['name']}")

for source in SOURCES:
    materialize_source(source)

[SKIP] FineWeb-Edu EN already complete.
[SKIP] FineWeb2-HQ PT already complete.
[SKIP] FineWeb2-HQ ES already complete.
[SKIP] OpenCoder FineWeb Code already complete.
[SKIP] OpenWebMath already complete.
[SKIP] peS2o Scientific already complete.


In [ ]:
# 7. CORPUS AUDIT
# Read-only integrity and semantic spot-check.

print("=" * 80)
print("MAIA CORPUS AUDIT")
print("=" * 80)

grand_total = 0
rng = random.Random(20260929)

for source in SOURCES:
    state, _carry = load_snapshot(source)
    manifest_path = MANIFEST_DIR / f"{source['id']}.json"

    assert state["complete"], f"Incomplete source: {source['name']}"
    assert state["tokens_written"] == source["target_tokens"]

    records = json.loads(manifest_path.read_text(encoding="utf-8"))

    physical_tokens = 0
    eos_count_sample = 0

    for record in records:
        path = TRAIN_DIR / record["shard"]
        assert path.exists()
        assert sha256_file(path) == record["sha256"]
        arr = np.memmap(path, dtype=DTYPE, mode="r")
        assert len(arr) == record["tokens"]
        assert len(arr) % SEQUENCE_LENGTH == 0
        assert int(arr.max()) < VOCAB_SIZE
        physical_tokens += len(arr)

    assert physical_tokens == source["target_tokens"]
    grand_total += physical_tokens

    # Decode one random 512-token window.
    record = rng.choice(records)
    path = TRAIN_DIR / record["shard"]
    arr = np.memmap(path, dtype=DTYPE, mode="r")
    if len(arr) > 512:
        start = rng.randrange(0, len(arr) - 512)
        sample = np.asarray(arr[start:start + 512], dtype=np.int64)
    else:
        sample = np.asarray(arr, dtype=np.int64)

    eos_count_sample = int(np.sum(sample == tokenizer.eos_token_id))

    print("\n" + "-" * 80)
    print(source["name"])
    print(f"tokens: {physical_tokens:,}")
    print(f"shards: {len(records):,}")
    print(f"EOS in decoded 512-token sample: {eos_count_sample}")
    print(tokenizer.decode(sample.tolist(), skip_special_tokens=False)[:2500])

assert grand_total == TARGET_TOTAL_TOKENS

print("\n" + "=" * 80)
print(f"TOTAL TOKENS: {grand_total:,}")
print("CORPUS AUDIT: PASS")
print("=" * 80)

MAIA CORPUS AUDIT

--------------------------------------------------------------------------------
FineWeb-Edu EN
tokens: 3,193,409,536
shards: 381
EOS in decoded 512-token sample: 0
 bedrock surface.
In the north part of the excavation area, beside the inner face of W103, a mass of small stones (L106; Figs. 5, 6), which fills a depression (depth c. 1.5 m) in the chalk bedrock was discovered. Upon removing part of the fill, it became apparent that it predated the road, but its nature was unclear.
Surface finds retrieved during the course of the excavation included an Iron Age II krater (Fig. 7:1) and jug (Fig. 7:2); a cooking pot (Fig. 7:3), a jar (Fig. 7:4) and jugs (Fig. 7:5, 6) from the late Hellenistic period; cooking pots (Fig. 7:7, 8) from the Early Roman period or possibly the late Hellenistic period; a fragment of a bag-shaped jar (Fig. 7:9) and a jug (table amphora? Fig. 7:10) from the Roman period; Late Roman–early Byzantine bowls (Fig. 7:11–15); a lid (Fig. 7:16) and a jar 

In [ ]:

# ==============================================================================
# 8. BUILD FROZEN VALIDATION CORPUS
# ==============================================================================
#
# PURPOSE
# -------
# Build a deterministic frozen validation corpus.
#
# Validation documents are selected by SHA-256 partitioning using the SAME
# holdout rule used by the training materializer.
#
# Important properties:
#
#   - deterministic
#   - disjoint from training
#   - explicit EOS after every document
#   - no document truncation
#   - fixed token budget
#   - uint16 storage
#   - resumable at source level
#
# Existing valid validation files are preserved.
#
# ==============================================================================

import os
import time
import hashlib
from pathlib import Path

import numpy as np
from datasets import load_dataset


# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

VALIDATION_TOTAL_TOKENS = 20_000 * SEQUENCE_LENGTH
# 20,480,000 tokens = 20,000 sequences of 1024 tokens


# ------------------------------------------------------------------------------
# Allocate validation tokens using the corpus mixture
# ------------------------------------------------------------------------------

def validation_target_tokens(index, source):

    total_sequences = (
        VALIDATION_TOTAL_TOKENS //
        SEQUENCE_LENGTH
    )

    if index < len(SOURCES) - 1:

        source_sequences = int(
            total_sequences *
            source["weight"]
        )

    else:

        used_sequences = sum(
            int(
                total_sequences *
                s["weight"]
            )
            for s in SOURCES[:-1]
        )

        source_sequences = (
            total_sequences -
            used_sequences
        )

    return (
        source_sequences *
        SEQUENCE_LENGTH
    )


# ------------------------------------------------------------------------------
# Fast tokenizer helper
# ------------------------------------------------------------------------------

def encode_document(text):

    ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if not ids:
        return None

    ids.append(
        tokenizer.eos_token_id
    )

    return np.asarray(
        ids,
        dtype=DTYPE,
    )


# ------------------------------------------------------------------------------
# Build validation corpus
# ------------------------------------------------------------------------------

print("=" * 80)
print("MAIA LITE - FROZEN VALIDATION CORPUS")
print("=" * 80)

validation_records = []

for source_index, source in enumerate(SOURCES):

    target = validation_target_tokens(
        source_index,
        source,
    )

    output_file = (
        VALIDATION_DIR /
        f"{source['id']}.bin"
    )

    expected_bytes = (
        target *
        np.dtype(DTYPE).itemsize
    )

    print()
    print("-" * 80)
    print(source["name"])
    print("-" * 80)

    print(
        f"Target tokens: "
        f"{target:,}"
    )


    # --------------------------------------------------------------------------
    # Preserve already completed validation files
    # --------------------------------------------------------------------------

    if (
        output_file.exists()
        and
        output_file.stat().st_size
        ==
        expected_bytes
    ):

        print(
            "[SKIP] Valid validation file already exists"
        )

        print(
            f"File: {output_file.name}"
        )

        print(
            f"SHA-256: "
            f"{sha256_file(output_file)}"
        )

        validation_records.append(
            (
                output_file,
                target,
            )
        )

        continue


    # --------------------------------------------------------------------------
    # Remove incomplete previous attempt
    # --------------------------------------------------------------------------

    temp_file = Path(
        str(output_file) +
        ".tmp"
    )

    if temp_file.exists():
        temp_file.unlink()

    if output_file.exists():
        print(
            "[INFO] Removing incomplete validation file"
        )

        output_file.unlink()


    # --------------------------------------------------------------------------
    # Open streaming source
    # --------------------------------------------------------------------------

    print()
    print("Opening source...")

    ds = load_dataset(
        source["repo"],
        source["config"],
        split=source["split"],
        streaming=True,
    )


    # --------------------------------------------------------------------------
    # Stream validation documents directly to disk
    #
    # This avoids accumulating millions of token IDs in Python lists.
    # --------------------------------------------------------------------------

    tokens_written = 0
    documents_scanned = 0
    documents_selected = 0

    start_time = time.time()

    with open(
        temp_file,
        "wb",
        buffering=16 * 1024 * 1024,
    ) as f:

        for row in ds:

            documents_scanned += 1

            text = row.get(
                source["text_field"]
            )

            if not isinstance(
                text,
                str,
            ):
                continue

            text = text.strip()

            if not text:
                continue


            # ------------------------------------------------------------------
            # EXACT SAME deterministic validation partition used by training.
            #
            # Do NOT change this independently from the training materializer.
            # ------------------------------------------------------------------

            if not is_validation_document(
                text
            ):
                continue


            # ------------------------------------------------------------------
            # Tokenize COMPLETE document.
            #
            # No truncation.
            # Explicit EOS.
            # ------------------------------------------------------------------

            token_array = encode_document(
                text
            )

            if token_array is None:
                continue

            documents_selected += 1


            # ------------------------------------------------------------------
            # Write only what is still required.
            # ------------------------------------------------------------------

            remaining = (
                target -
                tokens_written
            )

            if len(token_array) > remaining:

                token_array = (
                    token_array[:remaining]
                )


            token_array.tofile(
                f
            )

            tokens_written += len(
                token_array
            )


            # ------------------------------------------------------------------
            # Progress
            # ------------------------------------------------------------------

            if (
                documents_selected % 1000
                ==
                0
            ):

                elapsed = (
                    time.time() -
                    start_time
                )

                rate = (
                    tokens_written /
                    max(
                        elapsed,
                        1e-9,
                    )
                )

                percent = (
                    100.0 *
                    tokens_written /
                    target
                )

                print(
                    f"selected="
                    f"{documents_selected:,}  "
                    f"scanned="
                    f"{documents_scanned:,}  "
                    f"tokens="
                    f"{tokens_written:,}/"
                    f"{target:,}  "
                    f"({percent:.1f}%)  "
                    f"rate="
                    f"{rate:,.0f} tok/s"
                )


            if tokens_written >= target:
                break


    # --------------------------------------------------------------------------
    # Validate result
    # --------------------------------------------------------------------------

    if tokens_written < target:

        if temp_file.exists():
            temp_file.unlink()

        raise RuntimeError(
            f"Not enough validation tokens "
            f"for {source['name']}: "
            f"{tokens_written:,} / "
            f"{target:,}"
        )


    actual_bytes = (
        temp_file.stat().st_size
    )

    assert (
        actual_bytes
        ==
        expected_bytes
    ), (
        f"Validation size mismatch "
        f"for {source['name']}: "
        f"{actual_bytes:,} bytes "
        f"!= "
        f"{expected_bytes:,} bytes"
    )


    # --------------------------------------------------------------------------
    # Atomic publish
    # --------------------------------------------------------------------------

    os.replace(
        temp_file,
        output_file,
    )


    # --------------------------------------------------------------------------
    # Final source report
    # --------------------------------------------------------------------------

    elapsed = (
        time.time() -
        start_time
    )

    digest = sha256_file(
        output_file
    )

    print()
    print(
        f"Saved: "
        f"{output_file.name}"
    )

    print(
        f"Tokens: "
        f"{tokens_written:,}"
    )

    print(
        f"Documents selected: "
        f"{documents_selected:,}"
    )

    print(
        f"Documents scanned: "
        f"{documents_scanned:,}"
    )

    print(
        f"Elapsed: "
        f"{elapsed / 60:.2f} min"
    )

    print(
        f"SHA-256: "
        f"{digest}"
    )


    validation_records.append(
        (
            output_file,
            target,
        )
    )


# ------------------------------------------------------------------------------
# Global validation
# ------------------------------------------------------------------------------

validation_tokens = sum(
    tokens
    for _, tokens
    in validation_records
)

validation_sequences = (
    validation_tokens //
    SEQUENCE_LENGTH
)


print()
print("=" * 80)
print("FROZEN VALIDATION CORPUS COMPLETE")
print("=" * 80)

print(
    f"Files:      "
    f"{len(validation_records):,}"
)

print(
    f"Tokens:     "
    f"{validation_tokens:,}"
)

print(
    f"Sequences:  "
    f"{validation_sequences:,}"
)

print(
    f"Context:    "
    f"{SEQUENCE_LENGTH:,}"
)

assert (
    validation_tokens
    ==
    VALIDATION_TOTAL_TOKENS
)

assert (
    validation_sequences
    ==
    20_000
)

print()
print(
    "VALIDATION CORPUS: PASS"
)

MAIA LITE - FROZEN VALIDATION CORPUS

--------------------------------------------------------------------------------
FineWeb-Edu EN
--------------------------------------------------------------------------------
Target tokens: 9,216,000
[SKIP] Valid validation file already exists
File: fineweb_edu_en.bin
SHA-256: 9599f4fbc8ff89421e8a38ebcd125a0e1d8746732a79eb6de4f28e192280b08d

--------------------------------------------------------------------------------
FineWeb2-HQ PT
--------------------------------------------------------------------------------
Target tokens: 4,608,000
[SKIP] Valid validation file already exists
File: fineweb2_hq_pt.bin
SHA-256: 5807ef40ae6b6087164484afcc45da32cda3076246d70cc0a1a4115ceed4c6e5

--------------------------------------------------------------------------------
FineWeb2-HQ ES
--------------------------------------------------------------------------------
Target tokens: 4,608,000

Opening source...


Resolving data files:   0%|          | 0/1205 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/504 [00:00<?, ?it/s]

selected=1,000  scanned=194,971  tokens=827,713/4,608,000  (18.0%)  rate=4,490 tok/s
selected=2,000  scanned=387,122  tokens=1,555,561/4,608,000  (33.8%)  rate=4,259 tok/s
selected=3,000  scanned=587,205  tokens=2,498,223/4,608,000  (54.2%)  rate=4,626 tok/s
selected=4,000  scanned=778,896  tokens=3,169,404/4,608,000  (68.8%)  rate=4,504 tok/s
selected=5,000  scanned=973,876  tokens=3,831,293/4,608,000  (83.1%)  rate=4,425 tok/s
selected=6,000  scanned=1,171,320  tokens=4,537,569/4,608,000  (98.5%)  rate=4,372 tok/s

Saved: fineweb2_hq_es.bin
Tokens: 4,608,000
Documents selected: 6,140
Documents scanned: 1,194,511
Elapsed: 17.69 min
SHA-256: d3b0c5627f1184ba89692c3fbafafc68ceaec613492dc9acfe15c9753073ebc3

--------------------------------------------------------------------------------
OpenCoder FineWeb Code
--------------------------------------------------------------------------------
Target tokens: 1,024,000

Opening source...


Resolving data files:   0%|          | 0/510 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]


Saved: openwebmath.bin
Tokens: 614,400
Documents selected: 320
Documents scanned: 65,512
Elapsed: 0.41 min
SHA-256: c0d6f90524fbed7e38f7d03e7ef3b54fe4494b57c09b88f2ee123784e6d8d98a

--------------------------------------------------------------------------------
peS2o Scientific
--------------------------------------------------------------------------------
Target tokens: 409,600

Opening source...


Resolving data files:   0%|          | 0/102 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/102 [00:00<?, ?it/s]


Saved: pes2o.bin
Tokens: 409,600
Documents selected: 64
Documents scanned: 12,956
Elapsed: 0.55 min
SHA-256: be8f83dffe90e81ce368d71eff32082a2c20113034b8a7a2cb56f6e3b7eb25f2

FROZEN VALIDATION CORPUS COMPLETE
Files:      6
Tokens:     20,480,000
Sequences:  20,000
Context:    1,024

VALIDATION CORPUS: PASS


## Model initialization

The model is initialized **from scratch** with the GPT-2 Medium architecture. The parameter-count assertion is mandatory: training must not start unless it is exactly 354,823,168 parameters.


In [ ]:
# 8. INITIALIZE EXACT GPT-2 MEDIUM-MATCHED MAIA LITE

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=SEQUENCE_LENGTH,
    n_ctx=SEQUENCE_LENGTH,
    n_embd=1024,
    n_layer=24,
    n_head=16,
    n_inner=4096,
    activation_function="gelu_new",
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,
)

model = GPT2LMHeadModel(config)
parameter_count = sum(p.numel() for p in model.parameters())

print(f"Parameters: {parameter_count:,}")
print(config)

assert parameter_count == N_PARAMS_REFERENCE, (
    f"Parameter mismatch: {parameter_count:,} != {N_PARAMS_REFERENCE:,}"
)

print("\nExact GPT-2 Medium parameter match: PASS")

## Training

Training is deliberately separated from corpus construction. Before running this section:
- corpus audit must pass;
- frozen validation must be created;
- random decoded samples must be inspected manually.

Recommended behavioral gates: 1k, 5k, 10k, 25k, 50k, 100k, then regular checkpoints through 216,567.


In [ ]:

# ==============================================================================
# 9. STAGE CORPUS TO LOCAL SSD + BUILD TRAINING DATASET
# ==============================================================================
#
# The canonical materialized corpus lives on Google Drive.
#
# Trainer MUST NOT read training shards directly from Google Drive.
# At the beginning of every Colab runtime this cell:
#
#   1. Reads the canonical manifests.
#   2. Verifies the canonical shard inventory.
#   3. Copies all training shards to the local Colab SSD.
#   4. Verifies every local shard by size.
#   5. Builds MaiaBinaryDataset using ONLY local SSD paths.
#
# No streaming dataset is used by Trainer.
#
# ==============================================================================

import os
import json
import time
import shutil
import bisect
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import Dataset


# ==============================================================================
# LOCAL SSD CONFIGURATION
# ==============================================================================

LOCAL_ROOT = Path(
    "/content/maia_lite_v3"
)

LOCAL_TRAIN_DIR = (
    LOCAL_ROOT /
    "corpus" /
    "train"
)

LOCAL_TRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# BUILD CANONICAL RECORD LIST FROM MANIFESTS
# ==============================================================================

canonical_records = []

for source in SOURCES:

    manifest_path = (
        MANIFEST_DIR /
        f"{source['id']}.json"
    )

    assert manifest_path.exists(), (
        f"Missing manifest: "
        f"{manifest_path}"
    )

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    for record in manifest:

        canonical_path = (
            TRAIN_DIR /
            record["shard"]
        )

        tokens = int(
            record["tokens"]
        )

        assert canonical_path.exists(), (
            f"Missing canonical shard: "
            f"{canonical_path}"
        )

        canonical_records.append(
            (
                canonical_path,
                tokens,
            )
        )


# ==============================================================================
# VERIFY CANONICAL CORPUS
# ==============================================================================

dtype_size = np.dtype(
    DTYPE
).itemsize

canonical_tokens = sum(
    tokens
    for _, tokens
    in canonical_records
)

canonical_bytes = sum(
    path.stat().st_size
    for path, _
    in canonical_records
)


print("=" * 80)
print("MAIA LITE - TRAINING CORPUS STAGING")
print("=" * 80)

print()
print("CANONICAL CORPUS")

print(
    f"Shards: "
    f"{len(canonical_records):,}"
)

print(
    f"Tokens: "
    f"{canonical_tokens:,}"
)

print(
    f"Size:   "
    f"{canonical_bytes / 1024**3:.2f} GiB"
)


assert (
    canonical_tokens
    ==
    TARGET_TOTAL_TOKENS
), (
    f"Canonical token mismatch: "
    f"{canonical_tokens:,} != "
    f"{TARGET_TOTAL_TOKENS:,}"
)


for path, tokens in canonical_records:

    expected_bytes = (
        tokens *
        dtype_size
    )

    actual_bytes = (
        path.stat().st_size
    )

    assert (
        actual_bytes
        ==
        expected_bytes
    ), (
        f"Canonical shard size mismatch:\n"
        f"{path}\n"
        f"Expected: {expected_bytes:,}\n"
        f"Actual:   {actual_bytes:,}"
    )


print(
    "Canonical corpus: PASS"
)


# ==============================================================================
# CHECK LOCAL SSD CAPACITY
# ==============================================================================

disk = shutil.disk_usage(
    "/content"
)

required_bytes = int(
    canonical_bytes *
    1.10
)


print()
print("LOCAL SSD")

print(
    f"Destination: "
    f"{LOCAL_TRAIN_DIR}"
)

print(
    f"Free:        "
    f"{disk.free / 1024**3:.2f} GiB"
)

print(
    f"Required:    "
    f"{canonical_bytes / 1024**3:.2f} GiB"
)

print(
    f"With margin: "
    f"{required_bytes / 1024**3:.2f} GiB"
)


# Account for already copied local shards.
existing_local_bytes = sum(
    p.stat().st_size
    for p in LOCAL_TRAIN_DIR.glob("*.bin")
)

additional_required = max(
    0,
    canonical_bytes -
    existing_local_bytes
)


assert (
    disk.free
    >=
    additional_required
), (
    "Not enough local SSD space "
    "for the training corpus."
)


# ==============================================================================
# COPY CANONICAL SHARDS TO LOCAL SSD
# ==============================================================================

print()
print("=" * 80)
print("COPYING TRAINING CORPUS TO LOCAL SSD")
print("=" * 80)

start_time = time.time()

local_records = []

copied = 0
existing = 0
transferred_bytes = 0


for index, (
    canonical_path,
    tokens,
) in enumerate(
    canonical_records,
    start=1,
):

    local_path = (
        LOCAL_TRAIN_DIR /
        canonical_path.name
    )

    expected_bytes = (
        tokens *
        dtype_size
    )


    # --------------------------------------------------------------------------
    # Determine whether this shard must be copied
    # --------------------------------------------------------------------------

    needs_copy = True

    if local_path.exists():

        if (
            local_path.stat().st_size
            ==
            expected_bytes
        ):

            needs_copy = False

        else:

            local_path.unlink()


    # --------------------------------------------------------------------------
    # Copy atomically
    # --------------------------------------------------------------------------

    if needs_copy:

        temp_path = Path(
            str(local_path) +
            ".tmp"
        )

        if temp_path.exists():
            temp_path.unlink()

        shutil.copyfile(
            canonical_path,
            temp_path,
        )


        # Verify temporary copy
        actual_bytes = (
            temp_path.stat().st_size
        )

        assert (
            actual_bytes
            ==
            expected_bytes
        ), (
            f"Copy failed for "
            f"{canonical_path.name}"
        )


        # Atomic publication
        os.replace(
            temp_path,
            local_path,
        )

        copied += 1

        transferred_bytes += (
            expected_bytes
        )

    else:

        existing += 1


    # --------------------------------------------------------------------------
    # Final local verification
    # --------------------------------------------------------------------------

    assert local_path.exists()

    assert (
        local_path.stat().st_size
        ==
        expected_bytes
    )


    local_records.append(
        (
            local_path,
            tokens,
        )
    )


    # --------------------------------------------------------------------------
    # Progress
    # --------------------------------------------------------------------------

    if (
        index == 1
        or
        index % 50 == 0
        or
        index == len(
            canonical_records
        )
    ):

        elapsed = (
            time.time() -
            start_time
        )

        print(
            f"{index:4d}/"
            f"{len(canonical_records):4d}  "
            f"copied={copied:,}  "
            f"existing={existing:,}  "
            f"transferred="
            f"{transferred_bytes / 1024**3:.2f} GiB  "
            f"elapsed="
            f"{elapsed / 60:.1f} min"
        )


# ==============================================================================
# VERIFY COMPLETE LOCAL CORPUS
# ==============================================================================

local_tokens = sum(
    tokens
    for _, tokens
    in local_records
)

local_bytes = sum(
    path.stat().st_size
    for path, _
    in local_records
)


assert (
    local_tokens
    ==
    TARGET_TOTAL_TOKENS
)

assert (
    local_bytes
    ==
    canonical_bytes
)


for path, _ in local_records:

    assert str(path).startswith(
        "/content/maia_lite_v3/"
    ), (
        "ERROR: training shard "
        "is not on local SSD."
    )


print()
print("=" * 80)
print("LOCAL TRAINING CORPUS READY")
print("=" * 80)

print(
    f"Shards: "
    f"{len(local_records):,}"
)

print(
    f"Tokens: "
    f"{local_tokens:,}"
)

print(
    f"Size:   "
    f"{local_bytes / 1024**3:.2f} GiB"
)

print(
    f"Path:   "
    f"{LOCAL_TRAIN_DIR}"
)

print()
print(
    "TRAINING I/O: LOCAL SSD - PASS"
)


# ==============================================================================
# MAIA BINARY DATASET
# ==============================================================================

class MaiaBinaryDataset(Dataset):

    def __init__(
        self,
        records,
        sequence_length=1024,
    ):

        self.sequence_length = (
            sequence_length
        )

        self.entries = []
        self.cumulative = []

        total_sequences = 0


        for path, tokens in records:

            sequences = (
                tokens //
                sequence_length
            )

            if sequences <= 0:
                continue


            self.entries.append(
                (
                    Path(path),
                    tokens,
                    sequences,
                )
            )

            total_sequences += (
                sequences
            )

            self.cumulative.append(
                total_sequences
            )


        self.total_sequences = (
            total_sequences
        )

        self._maps = {}


    def __len__(self):

        return (
            self.total_sequences
        )


    def _memmap(
        self,
        path,
    ):

        key = str(
            path
        )

        if key not in self._maps:

            self._maps[key] = np.memmap(
                path,
                dtype=DTYPE,
                mode="r",
            )

        return (
            self._maps[key]
        )


    def __getitem__(
        self,
        index,
    ):

        if index < 0:
            index += self.total_sequences

        if (
            index < 0
            or
            index >= self.total_sequences
        ):

            raise IndexError(
                index
            )


        entry_idx = (
            bisect.bisect_right(
                self.cumulative,
                index,
            )
        )

        previous = (
            0
            if entry_idx == 0
            else
            self.cumulative[
                entry_idx - 1
            ]
        )

        local_sequence = (
            index -
            previous
        )


        path, _, _ = (
            self.entries[
                entry_idx
            ]
        )

        arr = self._memmap(
            path
        )


        start = (
            local_sequence *
            self.sequence_length
        )

        end = (
            start +
            self.sequence_length
        )


        ids_np = np.asarray(
            arr[start:end],
            dtype=np.int64,
        )


        assert (
            len(ids_np)
            ==
            self.sequence_length
        )


        ids = torch.from_numpy(
            ids_np
        ).long()


        return {
            "input_ids": ids,
            "attention_mask": torch.ones_like(
                ids
            ),
            "labels": ids.clone(),
        }


# ==============================================================================
# BUILD TRAINING DATASET FROM LOCAL SSD ONLY
# ==============================================================================

train_dataset = MaiaBinaryDataset(
    local_records,
    SEQUENCE_LENGTH,
)


# ==============================================================================
# FINAL DATASET ASSERTIONS
# ==============================================================================

training_sequences = len(
    train_dataset
)

training_tokens = (
    training_sequences *
    SEQUENCE_LENGTH
)


print()
print("=" * 80)
print("MAIA LITE TRAINING DATASET")
print("=" * 80)

print(
    f"Training sequences: "
    f"{training_sequences:,}"
)

print(
    f"Training tokens:    "
    f"{training_tokens:,}"
)


assert (
    training_tokens
    ==
    TARGET_TOTAL_TOKENS
), (
    f"Training dataset token mismatch: "
    f"{training_tokens:,} != "
    f"{TARGET_TOTAL_TOKENS:,}"
)


assert all(
    str(entry[0]).startswith(
        "/content/maia_lite_v3/"
    )
    for entry
    in train_dataset.entries
), (
    "ERROR: train_dataset contains "
    "non-local paths."
)


# ==============================================================================
# READ TEST
# ==============================================================================

test_item = train_dataset[0]

assert (
    test_item["input_ids"].shape[0]
    ==
    SEQUENCE_LENGTH
)

assert (
    test_item["labels"].shape[0]
    ==
    SEQUENCE_LENGTH
)

assert (
    test_item["attention_mask"].shape[0]
    ==
    SEQUENCE_LENGTH
)

assert (
    int(
        test_item[
            "input_ids"
        ].max()
    )
    <
    VOCAB_SIZE
)

assert (
    int(
        test_item[
            "input_ids"
        ].min()
    )
    >=
    0
)


print()
print(
    "Local SSD read test: PASS"
)

print(
    "Training dataset:    PASS"
)

MAIA LITE - TRAINING CORPUS STAGING

CANONICAL CORPUS
Shards: 849
Tokens: 7,096,467,456
Size:   13.22 GiB
Canonical corpus: PASS

LOCAL SSD
Destination: /content/maia_lite_v3/corpus/train
Free:        190.98 GiB
Required:    13.22 GiB
With margin: 14.54 GiB

COPYING TRAINING CORPUS TO LOCAL SSD
   1/ 849  copied=1  existing=0  transferred=0.02 GiB  elapsed=0.0 min
  50/ 849  copied=50  existing=0  transferred=0.78 GiB  elapsed=0.1 min
 100/ 849  copied=100  existing=0  transferred=1.56 GiB  elapsed=0.2 min
 150/ 849  copied=150  existing=0  transferred=2.34 GiB  elapsed=0.3 min
 200/ 849  copied=200  existing=0  transferred=3.12 GiB  elapsed=0.4 min
 250/ 849  copied=250  existing=0  transferred=3.91 GiB  elapsed=0.5 min
 300/ 849  copied=300  existing=0  transferred=4.69 GiB  elapsed=0.6 min
 350/ 849  copied=350  existing=0  transferred=5.47 GiB  elapsed=0.7 min
 400/ 849  copied=400  existing=0  transferred=6.25 GiB  elapsed=0.8 min
 450/ 849  copied=450  existing=0  transferred=7.0

In [ ]:

# ==============================================================================
# FINAL PREFLIGHT
# ==============================================================================
#
# This is the final integrity gate before configuring the Trainer.
#
# It verifies:
#   - tokenizer
#   - special tokens
#   - corpus token budget
#   - Chinchilla training budget
#   - training dataset size
#   - validation corpus size
#   - exact GPT-2 Medium parameter count
#   - local SSD training paths
#
# ==============================================================================

print("=" * 80)
print("MAIA LITE - FINAL PREFLIGHT")
print("=" * 80)


# ------------------------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------------------------

assert len(tokenizer) == VOCAB_SIZE == 50_257

assert tokenizer.bos_token_id == 0
assert tokenizer.pad_token_id == 1
assert tokenizer.eos_token_id == 2

print("Tokenizer:           PASS")


# ------------------------------------------------------------------------------
# Training corpus budget
# ------------------------------------------------------------------------------

source_target_tokens = sum(
    source["target_tokens"]
    for source in SOURCES
)

assert (
    source_target_tokens
    ==
    TARGET_TOTAL_TOKENS
)

assert (
    len(train_dataset)
    ==
    TARGET_STEPS *
    EFFECTIVE_BATCH
)

assert (
    len(train_dataset) *
    SEQUENCE_LENGTH
    ==
    TARGET_TOTAL_TOKENS
)

print("Training budget:     PASS")


# ------------------------------------------------------------------------------
# Local SSD enforcement
# ------------------------------------------------------------------------------

assert hasattr(
    train_dataset,
    "entries",
)

assert len(
    train_dataset.entries
) > 0

for path, _, _ in train_dataset.entries:

    path = Path(path)

    assert path.exists(), (
        f"Missing training shard: "
        f"{path}"
    )

    assert str(path).startswith(
        "/content/maia_lite_v3/"
    ), (
        "ERROR: training dataset is not "
        f"using local SSD: {path}"
    )

print("Training I/O:        LOCAL SSD - PASS")


# ------------------------------------------------------------------------------
# Validation corpus
# ------------------------------------------------------------------------------

validation_token_count = sum(
    tokens
    for _, tokens
    in validation_records
)

assert (
    validation_token_count
    ==
    VALIDATION_TOTAL_TOKENS
)

assert (
    VALIDATION_TOTAL_TOKENS
    %
    SEQUENCE_LENGTH
    ==
    0
)

validation_sequence_count = (
    VALIDATION_TOTAL_TOKENS //
    SEQUENCE_LENGTH
)

assert (
    validation_sequence_count
    ==
    20_000
)

print("Validation corpus:   PASS")


# ------------------------------------------------------------------------------
# Exact GPT-2 Medium architecture
# ------------------------------------------------------------------------------

actual_parameter_count = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

assert (
    actual_parameter_count
    ==
    N_PARAMS_REFERENCE
), (
    f"Parameter mismatch: "
    f"{actual_parameter_count:,} != "
    f"{N_PARAMS_REFERENCE:,}"
)

print("Model architecture:  PASS")


# ------------------------------------------------------------------------------
# Chinchilla ratio
# ------------------------------------------------------------------------------

tokens_per_parameter = (
    TARGET_TOTAL_TOKENS /
    N_PARAMS_REFERENCE
)


# ------------------------------------------------------------------------------
# Final report
# ------------------------------------------------------------------------------

print()
print("=" * 80)
print("FINAL PREFLIGHT: PASS")
print("=" * 80)

print(
    f"Parameters:          "
    f"{actual_parameter_count:,}"
)

print(
    f"Vocabulary:          "
    f"{len(tokenizer):,}"
)

print(
    f"Context length:      "
    f"{SEQUENCE_LENGTH:,}"
)

print(
    f"Training sequences:  "
    f"{len(train_dataset):,}"
)

print(
    f"Training tokens:     "
    f"{TARGET_TOTAL_TOKENS:,}"
)

print(
    f"Validation sequences:"
    f"  {validation_sequence_count:,}"
)

print(
    f"Validation tokens:   "
    f"{VALIDATION_TOTAL_TOKENS:,}"
)

print(
    f"Effective batch:     "
    f"{EFFECTIVE_BATCH:,} sequences"
)

print(
    f"Tokens / step:       "
    f"{EFFECTIVE_BATCH * SEQUENCE_LENGTH:,}"
)

print(
    f"Optimizer steps:     "
    f"{TARGET_STEPS:,}"
)

print(
    f"Tokens / parameter:  "
    f"{tokens_per_parameter:.6f}"
)

print()
print(
    "Training storage:    "
    "LOCAL SSD"
)

print(
    f"Training path:       "
    f"{LOCAL_TRAIN_DIR}"
)

print("=" * 80)

MAIA LITE - FINAL PREFLIGHT


NameError: name 'tokenizer' is not defined

In [ ]:

# ==============================================================================
# 10. TRAINING CONFIGURATION
# ==============================================================================
#
# MAIA LITE 355M - CLEAN PRETRAINING
#
# Exact GPT-2 Medium architecture:
#   Parameters:          354,823,168
#   Context:             1,024 tokens
#
# Training budget:
#   Effective batch:     32 sequences
#   Tokens / step:       32,768
#   Optimizer steps:     216,567
#   Total tokens:        7,096,467,456
#
# IMPORTANT
# ---------
# - Training data is materialized and fixed.
# - Trainer reads training data from LOCAL SSD.
# - Checkpoints are written to Google Drive.
# - Full checkpoint resume must restore model, optimizer,
#   scheduler, Trainer state, and RNG state.
# - NEVER use ignore_data_skip=True.
#
# ==============================================================================

import os
import torch

from transformers import TrainingArguments


# ------------------------------------------------------------------------------
# CUDA allocator
# ------------------------------------------------------------------------------

os.environ[
    "PYTORCH_CUDA_ALLOC_CONF"
] = "expandable_segments:True"


# ------------------------------------------------------------------------------
# Fundamental consistency checks
# ------------------------------------------------------------------------------

assert (
    MICRO_BATCH *
    GRAD_ACCUM
    ==
    EFFECTIVE_BATCH
), (
    "Effective batch mismatch."
)

assert (
    TARGET_STEPS *
    EFFECTIVE_BATCH *
    SEQUENCE_LENGTH
    ==
    TARGET_TOTAL_TOKENS
), (
    "Training token budget mismatch."
)

assert (
    len(train_dataset)
    ==
    TARGET_STEPS *
    EFFECTIVE_BATCH
), (
    "Training dataset length mismatch."
)


# ------------------------------------------------------------------------------
# Optimization configuration
# ------------------------------------------------------------------------------

LEARNING_RATE = 3e-4
WARMUP_STEPS = 2_000

WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0


# ------------------------------------------------------------------------------
# Precision
# ------------------------------------------------------------------------------

USE_BF16 = True
USE_FP16 = False
USE_TF32 = True


# ------------------------------------------------------------------------------
# Logging and checkpointing
# ------------------------------------------------------------------------------

LOGGING_STEPS = 25
SAVE_STEPS = 1_000


# ------------------------------------------------------------------------------
# TrainingArguments
# ------------------------------------------------------------------------------

training_args = TrainingArguments(

    # --------------------------------------------------------------------------
    # Persistent checkpoint directory
    # --------------------------------------------------------------------------

    output_dir=str(
        CHECKPOINT_DIR
    ),


    # --------------------------------------------------------------------------
    # Batch configuration
    # --------------------------------------------------------------------------

    per_device_train_batch_size=(
        MICRO_BATCH
    ),

    gradient_accumulation_steps=(
        GRAD_ACCUM
    ),


    # --------------------------------------------------------------------------
    # Exact Chinchilla-aligned training budget
    # --------------------------------------------------------------------------

    max_steps=(
        TARGET_STEPS
    ),


    # --------------------------------------------------------------------------
    # Optimizer / scheduler
    # --------------------------------------------------------------------------

    learning_rate=(
        LEARNING_RATE
    ),

    warmup_steps=(
        WARMUP_STEPS
    ),

    lr_scheduler_type="cosine",

    weight_decay=(
        WEIGHT_DECAY
    ),

    max_grad_norm=(
        MAX_GRAD_NORM
    ),


    # --------------------------------------------------------------------------
    # Precision
    # --------------------------------------------------------------------------

    bf16=(
        USE_BF16
    ),

    fp16=(
        USE_FP16
    ),

    tf32=(
        USE_TF32
    ),


    # --------------------------------------------------------------------------
    # Logging
    # --------------------------------------------------------------------------

    logging_steps=(
        LOGGING_STEPS
    ),


    # --------------------------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------------------------

    save_steps=(
        SAVE_STEPS
    ),

    save_total_limit=None,


    # --------------------------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------------------------

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    dataloader_drop_last=True,


    # --------------------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------------------

    seed=(
        SEED
    ),

    data_seed=(
        SEED
    ),


    # --------------------------------------------------------------------------
    # External reporting
    # --------------------------------------------------------------------------

    report_to="none",


    # --------------------------------------------------------------------------
    # Dataset
    # --------------------------------------------------------------------------

    remove_unused_columns=False,
)


# ------------------------------------------------------------------------------
# Configuration audit
# ------------------------------------------------------------------------------

assert (
    training_args.per_device_train_batch_size
    ==
    MICRO_BATCH
)

assert (
    training_args.gradient_accumulation_steps
    ==
    GRAD_ACCUM
)

assert (
    training_args.max_steps
    ==
    TARGET_STEPS
)

assert (
    abs(
        training_args.learning_rate -
        LEARNING_RATE
    )
    <
    1e-12
)

assert (
    training_args.warmup_steps
    ==
    WARMUP_STEPS
)


# ------------------------------------------------------------------------------
# Report
# ------------------------------------------------------------------------------

print("=" * 80)
print("MAIA LITE - TRAINING CONFIGURATION")
print("=" * 80)

print()

print(
    f"Transformers:          "
    f"{__import__('transformers').__version__}"
)

print(
    f"PyTorch:               "
    f"{torch.__version__}"
)

print()

print(
    f"Output directory:      "
    f"{CHECKPOINT_DIR}"
)

print()

print(
    f"Micro batch:           "
    f"{MICRO_BATCH:,}"
)

print(
    f"Gradient accumulation: "
    f"{GRAD_ACCUM:,}"
)

print(
    f"Effective batch:       "
    f"{EFFECTIVE_BATCH:,}"
)

print(
    f"Sequence length:       "
    f"{SEQUENCE_LENGTH:,}"
)

print(
    f"Tokens / step:         "
    f"{EFFECTIVE_BATCH * SEQUENCE_LENGTH:,}"
)

print()

print(
    f"Optimizer steps:       "
    f"{TARGET_STEPS:,}"
)

print(
    f"Total training tokens: "
    f"{TARGET_TOTAL_TOKENS:,}"
)

print()

print(
    f"Learning rate:         "
    f"{LEARNING_RATE:.2e}"
)

print(
    f"Warmup steps:          "
    f"{WARMUP_STEPS:,}"
)

print(
    "Scheduler:             "
    "cosine"
)

print(
    f"Weight decay:          "
    f"{WEIGHT_DECAY}"
)

print(
    f"Max gradient norm:     "
    f"{MAX_GRAD_NORM}"
)

print()

print(
    f"BF16:                  "
    f"{USE_BF16}"
)

print(
    f"FP16:                  "
    f"{USE_FP16}"
)

print(
    f"TF32:                  "
    f"{USE_TF32}"
)

print()

print(
    f"Logging every:         "
    f"{LOGGING_STEPS:,} steps"
)

print(
    f"Checkpoint every:      "
    f"{SAVE_STEPS:,} steps"
)

print()

print(
    "Training corpus:       "
    "LOCAL SSD"
)

print(
    "Checkpoints:           "
    "GOOGLE DRIVE"
)

print()

print(
    "TRAINING CONFIGURATION: PASS"
)

print("=" * 80)

ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [ ]:
# 11. TRAIN
# Uncomment only after all preflight audits and frozen validation are complete.

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
# )
#
# last_checkpoint = None
# checkpoints = sorted(
#     CHECKPOINT_DIR.glob("checkpoint-*"),
#     key=lambda p: int(p.name.split("-")[-1])
# )
# if checkpoints:
#     last_checkpoint = str(checkpoints[-1])
#     print("Resuming from:", last_checkpoint)
# else:
#     print("Starting from scratch.")
#
# trainer.train(resume_from_checkpoint=last_checkpoint)


## Final preflight before A100 training

Run all audits and the final preflight before uncommenting training. Recommended behavioral checkpoints: 1k, 5k, 10k, 25k, 50k, 100k, 150k, 200k, and 216,567.
